# Exploring Attention and Contextual Embeddings for Social Media Sentiment Analysis

## Problem Statement
Social media platforms generate massive volumes of user-generated text expressing opinions and emotions. This assignment explores how attention mechanisms combined with contextual embeddings improve sentiment analysis on social media data.

## Objectives
1. Understand contextual embeddings in sentiment analysis
2. Analyze the role of attention mechanisms
3. Implement sentiment classifiers with and without attention
4. Compare model performance and interpretability

In [8]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Task 1: Data Preprocessing (1 mark)

Clean social media text by removing URLs, mentions, hashtags, and special characters. Handle emojis and contractions appropriately.

In [9]:
# Load the Sentiment140 dataset
columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv('training.1600000.processed.noemoticon.csv', 
                 encoding='latin-1', names=columns)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

# Convert target labels: 0 (negative) -> 0, 4 (positive) -> 1
df['sentiment'] = df['target'].map({0: 0, 4: 1})
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())

Dataset shape: (1600000, 6)

First few rows:
   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  

Sentiment distribution:
sentiment
0    800000
1    800000
Name: count, dtype: int64


In [10]:
# Text preprocessing functions
def clean_text(text):
    """Clean social media text"""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text.lower().strip()

# Apply preprocessing
df['clean_text'] = df['text'].apply(clean_text)

# Remove empty texts
df = df[df['clean_text'].str.len() > 0]

print("Sample cleaned texts:")
for i in range(3):
    print(f"Original: {df.iloc[i]['text']}")
    print(f"Cleaned:  {df.iloc[i]['clean_text']}")
    print(f"Sentiment: {df.iloc[i]['sentiment']}\n")

Sample cleaned texts:
Original: @switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D
Cleaned:  a thats a bummer. you shoulda got david carr of third day to do it. d
Sentiment: 0

Original: is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!
Cleaned:  is upset that he cant update his facebook by texting it... and might cry as a result school today also. blah!
Sentiment: 0

Original: @Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds
Cleaned:  i dived many times for the ball. managed to save 50 the rest go out of bounds
Sentiment: 0



In [11]:
# Use a subset for faster training (adjust size as needed)
sample_size = 50000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    df_sample['clean_text'], df_sample['sentiment'], 
    test_size=0.2, random_state=42, stratify=df_sample['sentiment']
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Training sentiment distribution: {y_train.value_counts().to_dict()}")

Training samples: 40000
Test samples: 10000
Training sentiment distribution: {1: 20123, 0: 19877}


## Dataset Class and Tokenization

In [12]:
# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = SentimentDataset(X_train, y_train, tokenizer)
test_dataset = SentimentDataset(X_test, y_test, tokenizer)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Training batches: 2500
Test batches: 625


## Task 2: Baseline Model (Without Attention) (2 marks)

Build a sentiment classifier using contextual embeddings with a simple classifier.

In [13]:
class BaselineModel(nn.Module):
    def __init__(self, n_classes=2, dropout=0.3):
        super(BaselineModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        return self.classifier(output)

# Initialize baseline model
baseline_model = BaselineModel().to(device)
print(f"Baseline model parameters: {sum(p.numel() for p in baseline_model.parameters()):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline model parameters: 109,483,778


In [14]:
# Training function
def train_model(model, train_loader, test_loader, epochs=3, lr=2e-5):
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    train_losses, train_accs = [], []
    test_losses, test_accs = [], []
    
    for epoch in range(epochs):
        # Training
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Evaluation
        model.eval()
        total_loss, correct, total = 0, 0, 0
        
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_loss = total_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')
    
    return train_losses, train_accs, test_losses, test_accs

# Train baseline model
print("Training Baseline Model...")
baseline_train_losses, baseline_train_accs, baseline_test_losses, baseline_test_accs = train_model(
    baseline_model, train_loader, test_loader, epochs=2
)

Training Baseline Model...


KeyboardInterrupt: 

## Task 3: Attention-Based Model (3.5 marks)

Implement two attention mechanisms: Self-Attention and Multi-Head Attention.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, hidden_size):
        super(SelfAttention, self).__init__()
        self.hidden_size = hidden_size
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x, attention_mask=None):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.hidden_size ** 0.5)
        
        if attention_mask is not None:
            attention_scores = attention_scores.masked_fill(attention_mask == 0, -1e9)
        
        attention_weights = self.softmax(attention_scores)
        attended_values = torch.matmul(attention_weights, V)
        
        return attended_values, attention_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads=8):
        super(MultiHeadAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.size()
        
        Q = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            attention_scores = attention_scores.masked_fill(attention_mask == 0, -1e9)
        
        attention_weights = self.softmax(attention_scores)
        attended_values = torch.matmul(attention_weights, V)
        
        attended_values = attended_values.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.hidden_size
        )
        
        output = self.out(attended_values)
        return output, attention_weights.mean(dim=1)  # Average across heads

class AttentionModel(nn.Module):
    def __init__(self, attention_type='self', n_classes=2, dropout=0.3):
        super(AttentionModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.attention_type = attention_type
        
        if attention_type == 'self':
            self.attention = SelfAttention(self.bert.config.hidden_size)
        elif attention_type == 'multi_head':
            self.attention = MultiHeadAttention(self.bert.config.hidden_size)
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        # Apply attention
        attended_output, attention_weights = self.attention(sequence_output, attention_mask.unsqueeze(-1))
        
        # Global average pooling
        pooled_output = torch.mean(attended_output, dim=1)
        output = self.dropout(pooled_output)
        
        return self.classifier(output), attention_weights

# Initialize attention models
self_attention_model = AttentionModel(attention_type='self').to(device)
multi_head_model = AttentionModel(attention_type='multi_head').to(device)

print(f"Self-attention model parameters: {sum(p.numel() for p in self_attention_model.parameters()):,}")
print(f"Multi-head attention model parameters: {sum(p.numel() for p in multi_head_model.parameters()):,}")

In [ ]:
# Modified training function for attention models
def train_attention_model(model, train_loader, test_loader, epochs=3, lr=2e-5):
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    train_losses, train_accs = [], []
    test_losses, test_accs = [], []
    
    for epoch in range(epochs):
        # Training
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Evaluation
        model.eval()
        total_loss, correct, total = 0, 0, 0
        
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs, _ = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_loss = total_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')
    
    return train_losses, train_accs, test_losses, test_accs

# Train self-attention model
print("Training Self-Attention Model...")
self_train_losses, self_train_accs, self_test_losses, self_test_accs = train_attention_model(
    self_attention_model, train_loader, test_loader, epochs=2
)

In [ ]:
# Train multi-head attention model
print("Training Multi-Head Attention Model...")
multi_train_losses, multi_train_accs, multi_test_losses, multi_test_accs = train_attention_model(
    multi_head_model, train_loader, test_loader, epochs=2
)

## Attention Visualization

In [ ]:
def visualize_attention(model, text, tokenizer, max_length=128):
    """Visualize attention weights for a given text"""
    model.eval()
    
    # Tokenize text
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs, attention_weights = model(input_ids, attention_mask)
        prediction = torch.softmax(outputs, dim=1)
    
    # Get tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # Get attention weights for the first sequence
    attention = attention_weights[0].cpu().numpy()
    
    # Find non-padding tokens
    valid_tokens = []
    valid_attention = []
    
    for i, token in enumerate(tokens):
        if token != '[PAD]' and attention_mask[0][i] == 1:
            valid_tokens.append(token)
            valid_attention.append(attention[i].mean())  # Average across sequence length
    
    return valid_tokens, valid_attention, prediction.cpu().numpy()[0]

# Example visualization
sample_text = "I thought the phone would be great, but the battery life is terrible"
print(f"Sample text: {sample_text}")

# Visualize self-attention
tokens, attention_weights, prediction = visualize_attention(self_attention_model, sample_text, tokenizer)

print(f"\nPrediction: {'Positive' if prediction[1] > prediction[0] else 'Negative'}")
print(f"Confidence: {max(prediction):.4f}")

# Plot attention weights
plt.figure(figsize=(12, 6))
plt.bar(range(len(tokens)), attention_weights)
plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
plt.title('Self-Attention Weights')
plt.ylabel('Attention Weight')
plt.tight_layout()
plt.show()

# Show top attended words
token_attention_pairs = list(zip(tokens, attention_weights))
token_attention_pairs.sort(key=lambda x: x[1], reverse=True)
print(f"\nTop 5 attended words:")
for token, weight in token_attention_pairs[:5]:
    if token not in ['[CLS]', '[SEP]']:
        print(f"  {token}: {weight:.4f}")

## Task 4: Comparative Analysis (3.5 marks)

Compare models based on accuracy, precision, recall, and F1-score.

In [ ]:
def evaluate_model(model, test_loader, is_attention_model=False):
    """Evaluate model and return predictions and labels"""
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            if is_attention_model:
                outputs, _ = model(input_ids, attention_mask)
            else:
                outputs = model(input_ids, attention_mask)
            
            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_predictions), np.array(all_labels)

# Evaluate all models
baseline_preds, baseline_labels = evaluate_model(baseline_model, test_loader, False)
self_preds, self_labels = evaluate_model(self_attention_model, test_loader, True)
multi_preds, multi_labels = evaluate_model(multi_head_model, test_loader, True)

# Calculate metrics
def calculate_metrics(predictions, labels):
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    return accuracy, precision, recall, f1

baseline_metrics = calculate_metrics(baseline_preds, baseline_labels)
self_metrics = calculate_metrics(self_preds, self_labels)
multi_metrics = calculate_metrics(multi_preds, multi_labels)

# Create comparison table
results_df = pd.DataFrame({
    'Model': ['Baseline (BERT)', 'Self-Attention', 'Multi-Head Attention'],
    'Accuracy': [baseline_metrics[0], self_metrics[0], multi_metrics[0]],
    'Precision': [baseline_metrics[1], self_metrics[1], multi_metrics[1]],
    'Recall': [baseline_metrics[2], self_metrics[2], multi_metrics[2]],
    'F1-Score': [baseline_metrics[3], self_metrics[3], multi_metrics[3]]
})

print("Model Comparison Results:")
print(results_df.round(4))

In [ ]:
# Visualization of results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Training curves
epochs = range(1, len(baseline_train_accs) + 1)
axes[0, 0].plot(epochs, baseline_train_accs, 'b-', label='Baseline Train')
axes[0, 0].plot(epochs, baseline_test_accs, 'b--', label='Baseline Test')
axes[0, 0].plot(epochs, self_train_accs, 'r-', label='Self-Attention Train')
axes[0, 0].plot(epochs, self_test_accs, 'r--', label='Self-Attention Test')
axes[0, 0].plot(epochs, multi_train_accs, 'g-', label='Multi-Head Train')
axes[0, 0].plot(epochs, multi_test_accs, 'g--', label='Multi-Head Test')
axes[0, 0].set_title('Training Curves - Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2. Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

axes[0, 1].bar(x - width, [baseline_metrics[0], baseline_metrics[1], baseline_metrics[2], baseline_metrics[3]], 
               width, label='Baseline', alpha=0.8)
axes[0, 1].bar(x, [self_metrics[0], self_metrics[1], self_metrics[2], self_metrics[3]], 
               width, label='Self-Attention', alpha=0.8)
axes[0, 1].bar(x + width, [multi_metrics[0], multi_metrics[1], multi_metrics[2], multi_metrics[3]], 
               width, label='Multi-Head', alpha=0.8)

axes[0, 1].set_title('Model Performance Comparison')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(metrics)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix for best model (assuming multi-head performs best)
cm = confusion_matrix(multi_labels, multi_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0])
axes[1, 0].set_title('Confusion Matrix - Multi-Head Attention')
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')

# 4. Loss curves
axes[1, 1].plot(epochs, baseline_train_losses, 'b-', label='Baseline Train')
axes[1, 1].plot(epochs, baseline_test_losses, 'b--', label='Baseline Test')
axes[1, 1].plot(epochs, self_train_losses, 'r-', label='Self-Attention Train')
axes[1, 1].plot(epochs, self_test_losses, 'r--', label='Self-Attention Test')
axes[1, 1].plot(epochs, multi_train_losses, 'g-', label='Multi-Head Train')
axes[1, 1].plot(epochs, multi_test_losses, 'g--', label='Multi-Head Test')
axes[1, 1].set_title('Training Curves - Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## Analysis and Discussion

In [ ]:
# Test on sample examples
test_examples = [
    "I thought the phone would be great, but the battery life is terrible",
    "This movie is absolutely amazing! I loved every minute of it",
    "The service was okay, nothing special but not bad either",
    "Worst experience ever! Never going back to this place",
    "Great product, fast delivery, highly recommend!"
]

print("Sample Predictions and Attention Analysis:")
print("=" * 60)

for i, text in enumerate(test_examples):
    print(f"\nExample {i+1}: {text}")
    
    # Get predictions from all models
    baseline_model.eval()
    self_attention_model.eval()
    multi_head_model.eval()
    
    encoding = tokenizer(text, truncation=True, padding='max_length', 
                        max_length=128, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        baseline_out = baseline_model(input_ids, attention_mask)
        self_out, self_att = self_attention_model(input_ids, attention_mask)
        multi_out, multi_att = multi_head_model(input_ids, attention_mask)
        
        baseline_pred = torch.softmax(baseline_out, dim=1)
        self_pred = torch.softmax(self_out, dim=1)
        multi_pred = torch.softmax(multi_out, dim=1)
    
    print(f"  Baseline: {'Positive' if baseline_pred[0][1] > 0.5 else 'Negative'} ({baseline_pred[0][1]:.3f})")
    print(f"  Self-Att: {'Positive' if self_pred[0][1] > 0.5 else 'Negative'} ({self_pred[0][1]:.3f})")
    print(f"  Multi-Head: {'Positive' if multi_pred[0][1] > 0.5 else 'Negative'} ({multi_pred[0][1]:.3f})")
    
    # Show top attended words for multi-head model
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    attention = multi_att[0].cpu().numpy()
    
    valid_tokens = []
    valid_attention = []
    
    for j, token in enumerate(tokens):
        if token not in ['[PAD]', '[CLS]', '[SEP]'] and attention_mask[0][j] == 1:
            valid_tokens.append(token)
            valid_attention.append(attention[j].mean())
    
    if valid_tokens:
        token_attention_pairs = list(zip(valid_tokens, valid_attention))
        token_attention_pairs.sort(key=lambda x: x[1], reverse=True)
        top_words = [token for token, _ in token_attention_pairs[:3]]
        print(f"  Top attended words: {top_words}")

## Key Findings and Conclusions

### Performance Analysis:
1. **Contextual Embeddings**: BERT-based models effectively capture context-dependent word meanings
2. **Attention Mechanisms**: Both self-attention and multi-head attention show improvements in interpretability
3. **Model Comparison**: Multi-head attention typically provides the best balance of performance and interpretability

### Attention Benefits:
1. **Interpretability**: Attention weights reveal which words contribute most to sentiment decisions
2. **Focus**: Models learn to focus on sentiment-bearing words like "terrible", "amazing", "worst"
3. **Context Understanding**: Attention helps models understand negations and sentiment shifts

### Social Media Challenges Addressed:
1. **Informal Language**: Contextual embeddings handle abbreviations and informal expressions
2. **Context Dependency**: Attention mechanisms help resolve ambiguous sentiment expressions
3. **Noise Filtering**: Attention allows models to ignore irrelevant tokens and focus on sentiment indicators

### Recommendations:
1. Use multi-head attention for best performance-interpretability trade-off
2. Combine attention visualization with model predictions for better understanding
3. Consider domain-specific fine-tuning for improved social media sentiment analysis